# Reproduce the key results

Recomputes the headline numbers from the processed files with plain pandas/NumPy/statsmodels, independently of the web application. Run from the project root after `python manage.py import_datasets`. **Data limits:** state-level published aggregates (ecological); the causal estimate is an adjusted association unless assumptions A1-A7 hold.

In [1]:
import os, sys, json, numpy as np, pandas as pd, statsmodels.api as sm
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd())=='notebooks' else os.getcwd()
os.chdir(ROOT); sys.path.insert(0, ROOT)
a = pd.read_parquet('data/processed/state_analytic.parquet')
q = json.load(open('data/processed/quality_report.json'))
print(len(a), 'states; primary sample', int(a.in_primary_sample.sum()), '; analytic sha256', q['analytic_sha256'][:16])
assert a.state_code.notna().all()   # regression check: Nasarawa's code 'NA' must not become missing

37 states; primary sample 31 ; analytic sha256 8194ea464f22d4a3


## 1. Inequality in the primary out-of-school rate (MICS6 2021)

In [2]:
w = a.pop6_11_2015
y = a.oos_total_2021
mean_w = np.average(y, weights=w)
def gini(x, w):
    o = np.argsort(x.values); x = x.values[o]; w = w.values[o]/w.sum()
    cw = np.cumsum(w); cx = np.cumsum(x*w)/np.sum(x*w)
    return 1 - np.sum(w*(np.r_[0, cx[:-1]] + cx))
print('population-weighted mean OOS: %.2f' % mean_w)
print('Gini across states (weighted): %.4f' % gini(y, w))
print(a.groupby('region').apply(lambda d: np.average(d.oos_total_2021)).round(2))
print(a.sort_values('oos_total_2021', ascending=False)[['state','zone','oos_total_2021']].head(5).to_string(index=False))
print(a.sort_values('oos_total_2021')[['state','zone','oos_total_2021']].head(5).to_string(index=False))

population-weighted mean OOS: 23.01
Gini across states (weighted): 0.4769
region
North    36.00
South     6.12
dtype: float64
  state zone  oos_total_2021
  Kebbi   NW            65.0
Zamfara   NW            61.0
 Bauchi   NE            61.0
   Yobe   NE            58.0
  Gombe   NE            54.0
      state zone  oos_total_2021
        Imo   SE             1.0
      Ekiti   SW             2.0
    Bayelsa   SS             3.0
       Abia   SE             3.0
Cross River   SS             3.0


## 2. Adjusted association: log pupils per qualified teacher -> OOS rate

OLS with HC3 robust errors, adjusting for 2018 poverty and region (the identifying graph's backdoor set).

In [3]:
s = a[a.in_primary_sample].copy()
s['log_pqtr'] = np.log(s.pqtr); s['north'] = (s.region=='North').astype(float)
X = sm.add_constant(s[['log_pqtr','pov_215','north']])
m = sm.OLS(s.oos_total_2021, X).fit(cov_type='HC3', use_t=True)
b = m.params['log_pqtr']; lo, hi = m.conf_int().loc['log_pqtr']
k = np.log(0.9)
print('n =', int(m.nobs))
print('coef per log unit: %.3f  [%.3f, %.3f]' % (b, lo, hi))
print('effect of a 10%% cut in PQTR: %.2f pp  [%.2f, %.2f]' % (b*k, hi*k, lo*k))
naive = sm.OLS(s.oos_total_2021, sm.add_constant(s[['log_pqtr']])).fit(cov_type='HC3', use_t=True)
print('unadjusted (NOT causal): %.2f pp per 10%% cut' % (naive.params['log_pqtr']*k))

n = 31
coef per log unit: 4.064  [-4.814, 12.943]
effect of a 10% cut in PQTR: -0.43 pp  [-1.36, 0.51]
unadjusted (NOT causal): -1.77 pp per 10% cut


## 3. Hypothetical scenario S1 (cap PQTR at 35:1) - point estimate only; hypothetical

Illustration of the arithmetic, not a forecast. The interval for this effect includes zero.

In [4]:
t = a.copy(); t = t[t.in_primary_sample]
new_log = np.log(np.minimum(t.pqtr, 35))
change = np.maximum(0, t.oos_total_2021 + b*(new_log - np.log(t.pqtr))) - t.oos_total_2021   # a rate cannot fall below 0% (documented rule)
print('baseline (sample, weighted): %.2f' % np.average(t.oos_total_2021, weights=t.pop6_11_2015))
print('change at full implementation: %.2f pp' % np.average(change, weights=t.pop6_11_2015))

baseline (sample, weighted): 23.36
change at full implementation: -2.06 pp


## 4. Compare with stored application results

The web application's stored run should agree to numerical precision.

In [5]:
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE']='true'  # Jupyter runs an event loop; this notebook only reads
os.environ.setdefault('DJANGO_DEBUG','1'); os.environ.setdefault('DJANGO_SETTINGS_MODULE','eduineq.settings')
import django; django.setup()
from portal import services as S
C = S.latest_default('causal').results; est = C['estimates'][0]
print('stored OLS interval for log PQTR: [%.3f, %.3f]' % (est['ci_low'], est['ci_high']))
assert abs(est['ci_low']-lo) < 1e-6 and abs(est['ci_high']-hi) < 1e-6
print('notebook and application agree')
h = S.m.AnalysisRun.objects.filter(analysis_type='simulation', label__startswith='S1', status='succeeded').order_by('-created_at').first().results['headline']
print('S1 stored change: %.2f pp (notebook %.2f)' % (h['absolute_change_pp'], np.average(change, weights=t.pop6_11_2015)))
assert abs(h['absolute_change_pp'] - np.average(change, weights=t.pop6_11_2015)) < 5e-4

stored OLS interval for log PQTR: [-4.814, 12.943]
notebook and application agree
S1 stored change: -2.06 pp (notebook -2.06)
